In [ ]:
import os

import matplotlib.pyplot as plt
import duckdb
import numpy as np
import pandas as pd
import seaborn as sns
from shared import (
    DATA_FILES,
    LAT_MAX,
    LAT_MIN,
    LON_MAX,
    LON_MIN,
    get_con,
    setup_matplotlib,
)

BASE_DIR = os.path.abspath('.')
DATA_ROOT = os.path.join(BASE_DIR, 'DataGPS_parquet')
con = get_con(BASE_DIR)
setup_matplotlib()

print(f'BASE_DIR  : {BASE_DIR}')
print(f'DATA_ROOT : {DATA_ROOT}')
print(f'File data : {len(DATA_FILES)} bulan')

BASE_DIR  : /home/repal/Github/skripsi
DATA_ROOT : /home/repal/Github/skripsi/DataGPS_parquet
File data : 9 bulan


# deteksi maximum latitude longitude of DIY

In [ ]:
def get_max_lat_lon_diy(con: duckdb.DuckDBPyConnection, data_dir: str) -> dict:
    """
    Menghitung nilai maksimum latitude dan longitude dari data
    yang berada dalam bounding box DIY.
    
    Returns:
        dict: {'max_lat': float, 'max_lon': float, 'count': int}
    """
    
    # Build UNION ALL untuk semua file bulanan
    queries = []
    for name, filename in DATA_FILES:
        filepath = os.path.join(data_dir, filename).replace("\\", "/")
        queries.append(f"SELECT latitude, longitude FROM read_parquet('{filepath}')")
    
    union_query = " UNION ALL ".join(queries)
    
    # Query utama: filter wilayah DIY + aggregasi
    query = f"""
    WITH all_data AS (
        {union_query}
    ),
    diy_filtered AS (
        SELECT 
            latitude, 
            longitude
        FROM all_data
        WHERE latitude BETWEEN {LAT_MIN} AND {LAT_MAX}
          AND longitude BETWEEN {LON_MIN} AND {LON_MAX}
          AND latitude IS NOT NULL 
          AND longitude IS NOT NULL
    )
    SELECT 
        MAX(latitude) AS max_lat,
        MAX(longitude) AS max_lon,
        MIN(latitude) AS min_lat,
        MIN(longitude) AS min_lon,
        COUNT(*) AS valid_count
    FROM diy_filtered
    """
    
    result = con.execute(query).fetchdf().iloc[0]
    
    return {
        'max_lat': float(result['max_lat']),
        'max_lon': float(result['max_lon']),
        'min_lat': float(result['min_lat']),
        'min_lon': float(result['min_lon']),
        'count': int(result['valid_count'])
    }